In [1]:
# ============================================
# CELDA 1: IMPORTS
# ============================================

import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

from tqdm import tqdm

import os

In [2]:
# ============================================
# CELDA 2: CONFIGURACIÓN DE PATHS
# ============================================

FUSED_PATH = r"e:\Proyectos VS code\Hybrid-IT-OT-Intrusion-Detection\ProcesamientoData\output_phase2\fused_it_ot.csv"

OUTPUT_DIR = r"e:\Proyectos VS code\Hybrid-IT-OT-Intrusion-Detection\ProcesamientoData\output_phase3"

os.makedirs(OUTPUT_DIR, exist_ok=True)

TRANSFORMED_PATH = os.path.join(
    OUTPUT_DIR,
    "transformed_it_ot.csv"
)

print("INPUT:")
print(FUSED_PATH)

print("\nOUTPUT:")
print(TRANSFORMED_PATH)

INPUT:
e:\Proyectos VS code\Hybrid-IT-OT-Intrusion-Detection\ProcesamientoData\output_phase2\fused_it_ot.csv

OUTPUT:
e:\Proyectos VS code\Hybrid-IT-OT-Intrusion-Detection\ProcesamientoData\output_phase3\transformed_it_ot.csv


In [3]:
# ============================================
# CELDA 3: VALIDAR DATASET
# ============================================

if os.path.exists(FUSED_PATH):
    print("DATASET ENCONTRADO")
else:
    print("DATASET NO EXISTE")

# Limpiar output previo para evitar duplicados
if os.path.exists(TRANSFORMED_PATH):
    os.remove(TRANSFORMED_PATH)
    print("OUTPUT PREVIO ELIMINADO")

DATASET ENCONTRADO
OUTPUT PREVIO ELIMINADO


In [4]:
# ============================================
# CELDA 4: INSPECCIÓN INICIAL
# ============================================

sample_df = pd.read_csv(
    FUSED_PATH,
    nrows=5
)

print(sample_df.head())

print("\nCOLUMNAS:")
print(sample_df.columns)

                 timestamp                                  src_ip  src_port  \
0  2019-04-02 15:52:24.295                           192.168.1.103         0   
1  2019-04-02 15:52:36.962                             192.168.1.1     56504   
2  2019-04-02 15:52:25.294                           192.168.1.103         0   
3  2019-04-02 15:53:04.413  2405:6e00:10ce:2c00:20c:29ff:feee:e07a     21787   
4  2019-04-02 15:52:23.061                             192.168.1.1         0   

            dst_ip  dst_port  protocol  duration_ms  bidirectional_packets  \
0      224.0.0.253         0         2            0                      1   
1  239.255.255.250      1900        17           99                      7   
2      224.0.0.251         0         2            0                      1   
3  2600:1401:2::f0        53        17           22                      2   
4        224.0.0.1         0         2            0                      1   

   bidirectional_bytes  src2dst_packets  ...  src2

In [5]:
# ============================================
# CELDA 5: COLUMNAS CATEGÓRICAS (MODIFICADA)
# ============================================
# Sacamos 'traffic_label' de aquí para que NO use LabelEncoder
categorical_columns = [
    "application_name",
    "application_category",
    "device_type",
    "attack_type"
]
print("COLUMNAS CATEGÓRICAS A CODIFICAR (EXCLUYENDO TARGET):")
print(categorical_columns)

COLUMNAS CATEGÓRICAS A CODIFICAR (EXCLUYENDO TARGET):
['application_name', 'application_category', 'device_type', 'attack_type']


In [6]:
# ============================================
# CELDA 6: ENTRENAR LABEL ENCODERS
# ============================================

encoders = {}

for column in categorical_columns:

    print(f"\nENTRENANDO ENCODER: {column}")

    unique_values = set()

    reader = pd.read_csv(
        FUSED_PATH,
        usecols=[column],
        chunksize=100000
    )

    for chunk in reader:

        chunk[column] = chunk[column].astype(str)

        unique_values.update(
            chunk[column].unique()
        )

    encoder = LabelEncoder()

    encoder.fit(
        list(unique_values)
    )

    encoders[column] = encoder

    print(f"CLASES DETECTADAS: {len(encoder.classes_)}")


ENTRENANDO ENCODER: application_name
CLASES DETECTADAS: 208

ENTRENANDO ENCODER: application_category
CLASES DETECTADAS: 25

ENTRENANDO ENCODER: device_type
CLASES DETECTADAS: 70

ENTRENANDO ENCODER: attack_type
CLASES DETECTADAS: 7


In [7]:
# ============================================
# CELDA 7: COLUMNAS NUMÉRICAS
# ============================================

numeric_columns = [

    "src_port",
    "dst_port",
    "protocol",
    "duration_ms",
    "bidirectional_packets",
    "bidirectional_bytes",
    "src2dst_packets",
    "dst2src_packets",
    "src2dst_bytes",
    "dst2src_bytes",
    "src2dst_duration_ms",
    "dst2src_duration_ms",
    "log_length"

]

print("COLUMNAS NUMÉRICAS:")
print(numeric_columns)

COLUMNAS NUMÉRICAS:
['src_port', 'dst_port', 'protocol', 'duration_ms', 'bidirectional_packets', 'bidirectional_bytes', 'src2dst_packets', 'dst2src_packets', 'src2dst_bytes', 'dst2src_bytes', 'src2dst_duration_ms', 'dst2src_duration_ms', 'log_length']


In [8]:
# ============================================
# CELDA 8: ENTRENAR STANDARD SCALER
# ============================================

scaler = StandardScaler()

reader = pd.read_csv(
    FUSED_PATH,
    usecols=numeric_columns,
    chunksize=100000
)

partial_data = []

counter = 0

for chunk in reader:

    counter += 1

    print(f"SCALER CHUNK {counter}")

    partial_data.append(chunk)

    if counter == 10:
        break

partial_df = pd.concat(
    partial_data,
    ignore_index=True
)

scaler.fit(partial_df)

print("\nSCALER ENTRENADO")

SCALER CHUNK 1
SCALER CHUNK 2
SCALER CHUNK 3
SCALER CHUNK 4
SCALER CHUNK 5
SCALER CHUNK 6
SCALER CHUNK 7
SCALER CHUNK 8
SCALER CHUNK 9
SCALER CHUNK 10

SCALER ENTRENADO


In [9]:
# ============================================
# CELDA 9: TRANSFORMACIÓN COMPLETA (MODIFICADA)
# ============================================
first_chunk = True
reader = pd.read_csv(
    FUSED_PATH,
    chunksize=100000)
chunk_id = 0

for chunk in reader:
    chunk_id += 1
    print(f"\nPROCESANDO CHUNK {chunk_id}")

    # ----------------------------------------
    # TIMESTAMP
    # ----------------------------------------
    chunk["timestamp"] = pd.to_datetime(
        chunk["timestamp"],
        errors="coerce"
    )
    chunk["hour"] = chunk["timestamp"].dt.hour
    chunk["minute"] = chunk["timestamp"].dt.minute
    chunk["second"] = chunk["timestamp"].dt.second

    # ----------------------------------------
    # LABEL ENCODING (MANUAL Y AUTOMÁTICO)
    # ----------------------------------------
    
    # 1. MAPEO MANUAL PARA EL TARGET (Evita el orden alfabético defectuoso)
    # Forzamos: NORMAL (normal_pcaps) -> 0 | ATTACK (Cualquier ataque) -> 1
    chunk["traffic_label"] = chunk["traffic_label"].astype(str).str.strip()
    chunk["traffic_label"] = chunk["traffic_label"].map({
        "NORMAL": 0,
        "ATTACK": 1
    })
    
    # Control de seguridad: Si encuentra un valor inesperado, por defecto lo trata como ataque (1)
    chunk["traffic_label"] = chunk["traffic_label"].fillna(1).astype(int)


    # 2. CODIFICACIÓN AUTOMÁTICA (Para el resto de las variables categóricas)
    for column in categorical_columns:
        chunk[column] = chunk[column].astype(str)
        chunk[column] = encoders[column].transform(
            chunk[column]
        )

    # ----------------------------------------
    # STANDARD SCALER
    # ----------------------------------------
    chunk[numeric_columns] = scaler.transform(
        chunk[numeric_columns]
    )

    # ----------------------------------------
    # ELIMINAR COLUMNAS PESADAS
    # ----------------------------------------
    chunk.drop(
        columns=[
            "raw_log",
            "src_ip",
            "dst_ip",
            "timestamp"
        ],
        inplace=True
    )

    # ----------------------------------------
    # EXPORTAR
    # ----------------------------------------
    chunk.to_csv(
        TRANSFORMED_PATH,
        mode="a",
        header=first_chunk,
        index=False
    )

    first_chunk = False
    print(f"CHUNK {chunk_id} EXPORTADO")


PROCESANDO CHUNK 1
CHUNK 1 EXPORTADO

PROCESANDO CHUNK 2
CHUNK 2 EXPORTADO

PROCESANDO CHUNK 3
CHUNK 3 EXPORTADO

PROCESANDO CHUNK 4
CHUNK 4 EXPORTADO

PROCESANDO CHUNK 5
CHUNK 5 EXPORTADO

PROCESANDO CHUNK 6
CHUNK 6 EXPORTADO

PROCESANDO CHUNK 7
CHUNK 7 EXPORTADO

PROCESANDO CHUNK 8
CHUNK 8 EXPORTADO

PROCESANDO CHUNK 9
CHUNK 9 EXPORTADO

PROCESANDO CHUNK 10
CHUNK 10 EXPORTADO

PROCESANDO CHUNK 11
CHUNK 11 EXPORTADO

PROCESANDO CHUNK 12
CHUNK 12 EXPORTADO

PROCESANDO CHUNK 13
CHUNK 13 EXPORTADO

PROCESANDO CHUNK 14
CHUNK 14 EXPORTADO

PROCESANDO CHUNK 15
CHUNK 15 EXPORTADO

PROCESANDO CHUNK 16
CHUNK 16 EXPORTADO

PROCESANDO CHUNK 17
CHUNK 17 EXPORTADO

PROCESANDO CHUNK 18
CHUNK 18 EXPORTADO

PROCESANDO CHUNK 19
CHUNK 19 EXPORTADO

PROCESANDO CHUNK 20
CHUNK 20 EXPORTADO

PROCESANDO CHUNK 21
CHUNK 21 EXPORTADO

PROCESANDO CHUNK 22
CHUNK 22 EXPORTADO

PROCESANDO CHUNK 23
CHUNK 23 EXPORTADO

PROCESANDO CHUNK 24
CHUNK 24 EXPORTADO

PROCESANDO CHUNK 25
CHUNK 25 EXPORTADO

PROCESANDO CHUNK 

In [10]:
# ============================================
# CELDA 10: VALIDACIÓN FINAL
# ============================================

final_df = pd.read_csv(
    TRANSFORMED_PATH,
    nrows=5
)

print(final_df.head())

print("\nSHAPE:")
print(final_df.shape)

print("\nCOLUMNAS FINALES:")
print(final_df.columns)

   src_port  dst_port  protocol  duration_ms  bidirectional_packets  \
0 -2.740926 -0.353949 -1.153072    -0.159071              -0.006939   
1  0.833272 -0.209669  1.697311    -0.157960              -0.005903   
2 -2.740926 -0.353949 -1.153072    -0.159071              -0.006939   
3 -1.362775 -0.349924  1.697311    -0.158824              -0.006766   
4 -2.740926 -0.353949 -1.153072    -0.159071              -0.006939   

   bidirectional_bytes  src2dst_packets  dst2src_packets  src2dst_bytes  \
0            -0.016591        -0.004337        -0.006445      -0.010569   
1            -0.011866        -0.003011        -0.006445      -0.004041   
2            -0.016591        -0.004337        -0.006445      -0.010569   
3            -0.015823        -0.004337        -0.006132      -0.010413   
4            -0.016591        -0.004337        -0.006445      -0.010569   

   dst2src_bytes  ...  application_name  application_category  attack_type  \
0      -0.018081  ...               100     

In [11]:
# ============================================
# CELDA 11: DISTRIBUCIÓN FINAL
# ============================================

dist_df = pd.read_csv(
    TRANSFORMED_PATH,
    usecols=[
        "traffic_label",
        "attack_type"
    ]
)

print("\nTRAFFIC LABEL")
print(
    dist_df["traffic_label"].value_counts()
)

print("\nATTACK TYPE")
print(
    dist_df["attack_type"].value_counts()
)


TRAFFIC LABEL
traffic_label
1    3822077
0      85459
Name: count, dtype: int64

ATTACK TYPE
attack_type
2    1681665
6    1561405
0     475888
5      85459
4      71283
3      20782
1      11054
Name: count, dtype: int64


In [12]:
# ============================================
# CELDA 12: FINALIZACIÓN
# ============================================

print("===================================")
print("FASE 3 COMPLETADA")
print("===================================")

print("\nDATASET TRANSFORMADO:")
print(TRANSFORMED_PATH)

print("\nLISTO PARA:")

print("- SMOTE")
print("- PCA")
print("- TRAIN TEST SPLIT")
print("- XGBOOST")
print("- LIGHTGBM")
print("- DNN")
print("- STACKED ENSEMBLE")

FASE 3 COMPLETADA

DATASET TRANSFORMADO:
e:\Proyectos VS code\Hybrid-IT-OT-Intrusion-Detection\ProcesamientoData\output_phase3\transformed_it_ot.csv

LISTO PARA:
- SMOTE
- PCA
- TRAIN TEST SPLIT
- XGBOOST
- LIGHTGBM
- DNN
- STACKED ENSEMBLE
